In [91]:
import json
import os
import pandas as pd
import sqlite3

In [92]:
# Identify files with "employee" in the name and count them
file_count = sum("employee" in filename for filename in os.listdir("D:/Madhubalaji/Probability/employee"))

In [93]:
print(file_count)

3


In [94]:
# Identify and open the JSON file
json_file = [filename for filename in os.listdir("D:/Madhubalaji/Probability/employee") if filename.endswith('.json')][0]

In [95]:
# Read and parse JSON data
with open(os.path.join("D:/Madhubalaji/Probability/employee", json_file),"r" ) as f:
    data = json.load(f)

In [96]:
# Printing details of employee with id 8
for employee in data["objects"]:
    if employee["ID"] == "8":
        print("Employee details: ")
        print(employee)

Employee details: 
{'ID': '8', 'JobTitle': 'Audiologist', 'EmailAddress': 'Michaela_Little1010@yahoo.com', 'FirstNameLastName': 'Michaela Little', 'vaccinated': 'True'}


In [97]:
if "objects" in data:
    employee_key = "objects"
elif "employee" in data:
    employee_key = "employee"
else:
    raise KeyError("No key found for employee data")

In [98]:
print("keys in json data are : ", data.keys())

keys in json data are :  dict_keys(['objects'])


In [99]:
# Store column names
columns_name = list(data['objects'][0].keys())
print(columns_name)

['ID', 'JobTitle', 'EmailAddress', 'FirstNameLastName', 'vaccinated']


In [100]:
#sqlite  connection and create table

connection = sqlite3.connect('data.db')
cursor = connection.cursor()
cursor.execute("""CREATE TABLE IF NOT EXISTS employee (
                    ID INTEGER PRIMARY KEY,
                    JobTitle TEXT,
                    EmailAddress VARCHAR,
                    FirstNameLastName TEXT,
                    vaccinated BOOLEAN
                    )""")

In [102]:
employee_data = [(employee["ID"], employee['JobTitle'], employee["EmailAddress"], employee["FirstNameLastName"],employee["vaccinated"]) for employee in data["objects"]]
cursor.executemany('INSERT INTO employee VALUES (?,?,?,?,?)', employee_data)
connection.commit()

In [103]:
# Read data from SQLite into pandas DataFrame
df = pd.read_sql_query("SELECT * FROM employee", connection)

In [104]:
# Add bonus percent column
def calculate_bonus(row):
    if row['vaccinated'] == 1:
        return 0.15
    else:
        return 0.05

df['bonus_percent'] = df.apply(calculate_bonus, axis=1)

In [105]:
#Export DataFrame to Excel
df.to_excel("employee.xlsx", index=False)

In [106]:
# Close SQLite connection
connection.close()